# Analysis

The metrics of Section 3 for every model, format and bias, from the answers in `data/answers/tdms/`, plus the comparison
with CALM's LLM judges, the format comparisons, the variants, and time and cost. Tables are also saved as CSV in
`outputs/tables/` for the paper. Set `SMOKE` as in `03_runs`. No API calls.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
SMOKE = False
SMOKE_ITEMS = 1

In [ ]:
import sys
from pathlib import Path
CODE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE))   # judgebias and tdm
import json
import numpy as np
import pandas as pd
from judgebias import authority, design, metrics, reference, run
pd.set_option("display.width", 250); pd.set_option("display.max_rows", 400)
TABLES = CODE / "outputs" / "tables"; TABLES.mkdir(exist_ok=True)
FORMATS = ["calm", "native", "decomposed"]
short = lambda m: m.split("/")[1].replace("-experimental", "").replace(":free", "")

rows = design.build_design(citations=authority.load_citations())
if SMOKE:
    rows = design.smoke_sample(rows, n_items=SMOKE_ITEMS)
records = {(m, f): run.load_records(m, f, SMOKE) for m in run.MODELS for f in FORMATS}
# Mercury Decide is left out of the analysis: only 1,026 of its requests were answered (free tier, 1,000 per day).
# Each bias pools CALM's versions only; our additions (citation on the better response, neutral note) are tests of their own.
tables = {k: metrics.separate_additions(metrics.table(rows, v)) for k, v in records.items() if v and "mercury" not in k[0]}
print(len(tables), "model-format runs loaded")

24 model-format runs loaded


## Check: does the statistics find planted biases, and only those?

In [ ]:
full = design.build_design(citations=authority.load_citations())
def simulate(seed):
    """A judge with P(R1) about 0.8 and a planted bandwagon bias of 0.3; nothing else is biased."""
    rng = np.random.default_rng(seed); recs = []
    for r in full:
        for rep in ([0, 1] if r["bias"] == "original" else [0]):
            p_r1 = float(np.clip(0.8 - (0.3 if r["bias"] == "bandwagon" else 0) + rng.normal(0, 0.15), 0, 1))
            p_a = p_r1 if r["r1_label"] == "A" else 1 - p_r1
            recs.append({"id": r["id"], "repeat": rep, "choice": "A" if p_a > 0.5 else "B", "p_A": p_a})
    return metrics.table(full, recs)
hits = alarms = 0
for seed in range(20):
    res_sim = metrics.summarize(simulate(seed), n_boot=1000, seed=seed)
    hits += int(((res_sim.p_bias < 0.05) & (res_sim.bias == "bandwagon")).sum())
    alarms += int(((res_sim.p_bias < 0.05) & (res_sim.bias != "bandwagon")).sum())
print(f"planted bias found in {hits} of 20 runs; false alarms in {alarms} of {20 * 8} unbiased tests")

planted bias found in 20 of 20 runs; false alarms in 2 of 160 unbiased tests


## Completeness: failed requests and 50/50 answers

A 50/50 answer has no verdict (Section 3 of the paper): it counts for the probability shifts but not for RR and CR.

In [ ]:
comp = []
for (m, f), recs in records.items():
    bad = [r for r in recs if not r["ok"]]
    comp.append({"model": short(m), "format": f, "requests": len(recs), "failed": len(bad),
                 "50/50 answers": sum(r["ok"] and abs(r["p_A"] - 0.5) < metrics.TIE for r in recs),
                 "example error": (bad[0]["error"] or "")[:90] if bad else ""})
comp = pd.DataFrame(comp)
comp[(comp.failed > 0) | (comp["50/50 answers"] > 0)]

,model,format,requests,failed,50/50 answers,example error
0,jev-1.13,calm,15195,0,65,
1,jev-1.13,native,15195,0,70,
2,jev-1.13,decomposed,15195,0,17,
3,kev-4b,calm,15195,0,1,
4,kev-4b,native,15195,0,5,
5,kev-4b,decomposed,15195,0,2,
6,tev1-4b,calm,15195,1,414,"HTTP 502: {""error"":{""message"":""Tev1 answered \..."
7,tev1-4b,native,15195,0,307,
8,tev1-4b,decomposed,15195,0,33,
9,mercury-decide,calm,14695,13669,11,"HTTP 429: {""error"":{""message"":""Rate limit exce..."


## 50/50 answers: which option does the API return?

A typed model can answer 50/50, and the reply then still names one option in its `choice` field. TypeSafe's
documentation defines `choice` as "the option with the highest probability" and says nothing about ties; OpenRouter's
API reference only lists the fields. The table shows, for the two single-question formats, how often each model
answers 50/50 and which option the API then returns. `distinct probabilities` shows how finely a model reports
P(A): Jev, for example, rounds to two decimals.

In [ ]:
ties = []
for (m, f), recs in records.items():
    if f == "decomposed" or "mercury" in m:
        continue
    ok = [r for r in recs if r["ok"]]
    t = [r for r in ok if abs(r["p_A"] - 0.5) < metrics.TIE]
    ties.append({"model": short(m), "format": f, "answers": len(ok), "50/50": len(t), "share": len(t) / len(ok),
                 "choice A": sum(r["choice"] == "A" for r in t), "choice B": sum(r["choice"] == "B" for r in t),
                 "distinct probabilities": len({round(r["p_A"], 6) for r in ok})})
ties = pd.DataFrame(ties)
ties.to_csv(TABLES / "ties.csv", index=False)
ties.round(4)

,model,format,answers,50/50,share,choice A,choice B,distinct probabilities
0,jev-1.13,calm,15195,65,0.0043,32,33,101
1,jev-1.13,native,15195,70,0.0046,37,33,101
2,kev-4b,calm,15195,1,0.0001,1,0,7563
3,kev-4b,native,15195,5,0.0003,1,4,7560
4,tev1-4b,calm,15194,414,0.0272,414,0,319
5,tev1-4b,native,15195,307,0.0202,307,0,153
6,solar-decide,calm,15195,117,0.0077,117,0,139
7,solar-decide,native,15195,117,0.0077,117,0,143
8,d1,calm,15195,251,0.0165,251,0,241
9,d1,native,15195,263,0.0173,263,0,237


### The reported confidence

TypeSafe's documentation (Jev's maker; the other providers were not checked) defines the confidence of a Choice with n options as (p_max - 1/n) / (1 - 1/n), which is
2 p_max - 1 for our two options and 0 for a 50/50 answer. We do not use the confidence, but it shows whether the
models fill in the same field the same way. Checked on the first 500 answers of each model in the CALM format plus up to 50
further 50/50 answers, read from the cache (no API calls), against three rules: the documented one, p_max itself, and 1 minus the entropy of
the two probabilities (tolerance 0.015, because Jev rounds both the probabilities and the
confidence to two decimals).

In [ ]:
import os
from tdm import DecisionClient
from tdm.config import CACHE_DIR
# Reads the raw API answers from the cache (cache/decisions.sqlite), which is not part of the repository,
# so without it this check is skipped. The output below is from our run.
HAVE_CACHE = (CACHE_DIR / "decisions.sqlite").exists() and bool(os.environ.get("OPENROUTER_API_KEY"))
by_id = {r["id"]: r for r in rows}
conf = []
for m in (run.MODELS if HAVE_CACHE else []):
    if "mercury" in m:
        continue
    client = DecisionClient(m)
    ok = [x for x in records[(m, "calm")] if x["ok"]]
    sample = ok[:500] + [x for x in ok[500:] if abs(x["p_A"] - 0.5) < metrics.TIE][:50]
    answers = [run.cached_answers(client, by_id[r["id"]], "calm", r["repeat"]) for r in sample]
    if None in answers:
        continue
    answers = [a["verdict"] for a in answers]
    p = np.array([max(a["probabilities"].values()) for a in answers])
    c = np.array([a["confidence"] for a in answers])
    entropy = -(p * np.log2(p) + (1 - p) * np.log2(np.clip(1 - p, 1e-12, 1)))
    conf.append({"model": short(m), "checked": len(answers),
                 "= 2 p_max - 1 (documented)": int((abs(c - (2 * p - 1)) < 0.015).sum()),
                 "= p_max": int((abs(c - p) < 0.015).sum()), "= 1 - entropy": int((abs(c - (1 - entropy)) < 0.015).sum()),
                 "confidence when p_max = 0.5": sorted(set(c[abs(p - 0.5) < metrics.TIE].round(3)))})
pd.DataFrame(conf)

,model,checked,= 2 p_max - 1 (documented),= p_max,= 1 - entropy,confidence when p_max = 0.5
0,jev-1.13,550,550,112,116,"[0.0, 0.01]"
1,kev-4b,501,501,1,16,[0.0]
2,tev1-4b,550,6,550,0,[0.5]
3,solar-decide,550,77,16,550,[0.0]
4,d1,550,550,28,56,[0.0]
5,pplx-decider-v1.1-27b,500,500,62,2,[]
6,clef,500,45,23,7,[]
7,clef-flash,500,1,0,36,[]


## Main results: one row per model, format and test

**How to read the table.** Each row is one test of one model in one format. Each bias pools CALM's versions only; our
two additions (a citation on the better response, the neutral identity note) are rows of their own.

* `n`: perturbed prompts that count (all three answers have a verdict; 50/50 answers drop out).
* `RR`: share of verdicts that survive the perturbation. `CR`: share that survive a plain repeat of the original.
* `excess` = CR - RR: verdicts changed by the perturbation beyond those that change anyway. 0 = no bias, 1 = every
  verdict flips. Negative = the perturbation changes fewer verdicts than a plain repeat (it makes the judge steadier).
* `significant`: the permutation test (Benjamini-Hochberg adjusted p < 0.05) says the perturbation changes verdicts
  more often than repetition, i.e. the model shows this bias. It says nothing about size: read `excess` and `shift`.
* `shift`: mean change in the probability of the response the perturbation targets (+ = pulled towards it).
  Targets: bandwagon the worse response the majority backs; compassion fade the worse response, which gets the strong
  model's name; authority, sentiment and the notes the response they are put on; verbosity the padded (worse) one;
  fallacy the broken (better) one; position the response shown first in the original, which moves to second place
  (so + = a preference for the second position).
* `noise_floor`: mean change in that probability on a plain repeat, for scale.
* `acc_before` / `acc_after`: share of verdicts for the better response on the original and on the perturbed prompts.
* **Fallacy oversight is reversed**: the better response's reasoning is broken, so changing the verdict is correct.
  Its RR is near 0 and its excess near 1 for good judges; we report `correct_share` = 1 - RR in its own table.

In [ ]:
res = pd.concat([metrics.summarize(df, by=["bias"]).assign(model=short(m), format=f)
                 for (m, f), df in tables.items()], ignore_index=True)
res["significant"], res["p_adjusted"] = metrics.adjust(res.p_bias)
res.to_csv(TABLES / "main.csv", index=False)
print("Bias tests (fallacy oversight is shown separately below)")
display(res[res.bias != "fallacy"][["model", "format", "bias", "n", "RR", "RR_lo", "RR_hi", "CR", "excess", "excess_lo",
     "excess_hi", "p_adjusted", "significant", "shift", "shift_lo", "shift_hi", "noise_floor", "acc_before", "acc_after"]].round(3))
print("Fallacy oversight: a changed verdict is the correct reaction. correct_share = 1 - RR; excess = RR + CR - 1,")
print("the unchanged verdicts beyond chance, so that excess is the bias beyond chance in every test")
res[res.bias == "fallacy"][["model", "format", "n", "correct_share", "CR", "excess", "excess_lo", "excess_hi", "shift", "shift_lo", "shift_hi",
                             "acc_before", "acc_after"]].round(3)

Bias tests (fallacy oversight is shown separately below)


,model,format,bias,n,RR,RR_lo,RR_hi,CR,excess,excess_lo,excess_hi,p_adjusted,significant,shift,shift_lo,shift_hi,noise_floor,acc_before,acc_after
0,jev-1.13,calm,authority,449,0.871,0.820,0.918,0.993,0.122,0.076,0.171,0.000,True,-0.146,-0.169,-0.124,0.015,0.722,0.851
1,jev-1.13,calm,"authority, better response",447,0.702,0.636,0.767,0.993,0.291,0.224,0.359,0.000,True,-0.210,-0.239,-0.181,0.015,0.718,0.421
2,jev-1.13,calm,bandwagon,1751,0.781,0.743,0.819,0.982,0.200,0.161,0.240,0.000,True,0.175,0.164,0.186,0.013,0.776,0.557
3,jev-1.13,calm,compassion,436,0.963,0.945,0.979,0.984,0.021,0.002,0.039,0.051,False,0.018,0.013,0.023,0.013,0.780,0.766
4,jev-1.13,calm,distraction,872,0.928,0.910,0.945,0.982,0.054,0.035,0.073,0.000,True,-0.057,-0.063,-0.052,0.013,0.779,0.766
5,jev-1.13,calm,diversity,5236,0.959,0.947,0.971,0.983,0.023,0.011,0.036,0.002,True,-0.029,-0.032,-0.026,0.013,0.779,0.769
6,jev-1.13,calm,"diversity, neutral note",871,0.951,0.935,0.965,0.982,0.031,0.017,0.045,0.004,True,-0.032,-0.037,-0.028,0.013,0.778,0.766
8,jev-1.13,calm,position,437,0.897,0.868,0.924,0.982,0.085,0.055,0.115,0.000,True,0.043,0.031,0.056,0.013,0.776,0.769
9,jev-1.13,calm,sentiment,1190,0.634,0.610,0.660,1.000,0.366,0.340,0.390,0.000,True,-0.338,-0.356,-0.321,0.004,1.000,0.634
10,jev-1.13,calm,verbosity,500,1.000,1.000,1.000,1.000,0.000,0.000,0.000,1.000,False,-0.020,-0.024,-0.016,0.004,0.996,0.996


Fallacy oversight: a changed verdict is the correct reaction. correct_share = 1 - RR; excess = RR + CR - 1,
the unchanged verdicts beyond chance, so that excess is the bias beyond chance in every test


,model,format,n,correct_share,CR,excess,excess_lo,excess_hi,shift,shift_lo,shift_hi,acc_before,acc_after
7,jev-1.13,calm,498,0.980,1.000,0.020,0.008,0.034,-0.925,-0.935,-0.914,0.996,0.016
19,jev-1.13,native,500,0.984,0.998,0.014,0.004,0.026,-0.921,-0.931,-0.910,0.996,0.012
31,jev-1.13,decomposed,500,0.984,1.000,0.016,0.006,0.028,-0.892,-0.901,-0.881,0.996,0.012
43,kev-4b,calm,500,0.990,1.000,0.010,0.002,0.020,-0.715,-0.726,-0.704,0.990,0.000
55,kev-4b,native,499,0.966,0.998,0.032,0.018,0.048,-0.640,-0.652,-0.628,0.972,0.006
67,kev-4b,decomposed,500,0.946,1.000,0.054,0.036,0.074,-0.628,-0.640,-0.616,0.952,0.006
79,tev1-4b,calm,495,0.929,1.000,0.071,0.049,0.093,-0.714,-0.730,-0.698,0.988,0.059
91,tev1-4b,native,494,0.923,1.000,0.077,0.055,0.101,-0.744,-0.760,-0.727,0.974,0.051
103,tev1-4b,decomposed,498,0.924,1.000,0.076,0.054,0.100,-0.741,-0.757,-0.725,0.976,0.052
115,solar-decide,calm,500,0.924,0.982,0.058,0.034,0.084,-0.844,-0.863,-0.823,0.956,0.032


In [ ]:
# Overview in the CALM format: robustness rate per bias and model (* = significant after Benjamini-Hochberg)
def star(r):
    return f"{r.RR:.2f}{'*' if r.significant else ''}"
calm = res[res.format == "calm"].copy()
calm["cell"] = calm.apply(star, axis=1)
print("Robustness rate, CALM format"); display(calm.pivot(index="bias", columns="model", values="cell"))
print("Probability shift of the targeted response, CALM format")
display(calm.pivot(index="bias", columns="model", values="shift").round(3))

Robustness rate, CALM format


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
bias,,,,,,,,
authority,0.83*,0.88*,0.82*,0.87*,0.85*,0.84*,0.81*,0.85*
"authority, better response",0.66*,0.92*,0.70*,0.70*,0.78*,0.62*,0.78*,0.77*
bandwagon,0.73*,0.88*,0.93*,0.78*,0.81*,0.74*,0.87*,0.96*
compassion,0.95*,0.93*,0.92*,0.96,0.85*,0.96*,0.83*,0.79*
distraction,0.99,0.96*,0.97*,0.93*,0.94*,0.98*,0.88,0.95*
diversity,0.98*,0.95*,0.95*,0.96*,0.91*,0.98*,0.89,0.94*
"diversity, neutral note",0.98*,0.95*,0.95*,0.95*,0.95*,0.98*,0.88*,0.95*
fallacy,0.03,0.12,0.04,0.02,0.01,0.04,0.08,0.07
position,0.86*,0.81*,0.87*,0.90*,0.81*,0.92*,0.76*,0.76*


Probability shift of the targeted response, CALM format


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
bias,,,,,,,,
authority,-0.158,-0.028,-0.160,-0.146,-0.076,-0.185,-0.093,-0.089
"authority, better response",-0.310,-0.035,-0.198,-0.210,-0.100,-0.254,-0.150,-0.136
bandwagon,0.262,0.109,0.045,0.175,0.085,0.189,0.055,0.038
compassion,0.043,0.045,0.012,0.018,0.066,0.024,0.067,0.109
distraction,-0.008,-0.021,-0.004,-0.057,-0.028,-0.006,-0.009,-0.031
diversity,-0.020,-0.042,-0.013,-0.029,-0.042,-0.011,-0.009,-0.046
"diversity, neutral note",-0.011,-0.042,-0.013,-0.032,-0.020,-0.009,-0.013,-0.038
fallacy,-0.926,-0.792,-0.824,-0.925,-0.715,-0.853,-0.844,-0.714
position,0.083,0.101,-0.079,0.043,0.082,-0.012,-0.120,0.065


## Comparison with CALM's LLM judges

CALM's numbers come from six LLMs at temperature 0.7, whose consistency rates are lower (0.86 to 0.93 on the
alignment set). Besides the raw robustness rate we therefore compare the *excess* of changed verdicts over
repetition, CR - RR, which removes the part of the changes that repetition alone causes. Our numbers use the CALM
format.

* Position, verbosity, compassion fade and fallacy oversight: CALM reports one number per judge (Table 4).
* Bandwagon, authority, sentiment, distraction and diversity: CALM breaks them down (Table 8), so we compare variant
  by variant: sentiment per tone and side, bandwagon per majority share, distraction per side, authority per citation
  type on the worse response (as in CALM's example), diversity per group with both sides pooled (CALM does not say
  which side it used).

In [ ]:
ONE_NUMBER = ["position", "verbosity", "compassion", "fallacy"]
ours, llm = [], []
for (m, f), df in tables.items():
    if f != "calm":
        continue
    for r in metrics.summarize(df[df.bias.isin(ONE_NUMBER)], by=["bias"], n_boot=2000).itertuples():
        ours.append({"judge": short(m), "kind": "typed", "bias": r.bias, "variant": "-",
                     "value": 1 - r.RR if r.bias == "fallacy" else r.RR, "CR": r.CR})
llm += [dict(x, kind="LLM (CALM)", variant="-") for x in reference.table4_long() if x["bias"] in ONE_NUMBER]

def our_variants(bias, name):
    """Our variants that match one of CALM's Table 8 rows."""
    if bias == "authority":
        return [f"{name}-on-R2"]                      # CALM's example puts the citation on the worse response
    if bias == "diversity":
        return [f"{name}-on-R1", f"{name}-on-R2"]     # CALM does not say which side; both pooled
    return [name]

for (bias, calm_name, name), values in reference.TABLE8.items():
    cr = reference.TABLE4["CR_fact" if bias in reference.FACT_BIASES else "CR_alignment"]
    llm += [{"judge": j, "kind": "LLM (CALM)", "bias": bias, "variant": calm_name, "value": v, "CR": c_}
            for j, v, c_ in zip(reference.JUDGES, values, cr)]
    for (m, f), df in tables.items():
        if f != "calm":
            continue
        s = metrics.summarize(df[(df.bias == bias) & df.variant.isin(our_variants(bias, name))], by=["bias"], n_boot=500).iloc[0]
        ours.append({"judge": short(m), "kind": "typed", "bias": bias, "variant": calm_name, "value": s.RR, "CR": s.CR})

comparison = pd.DataFrame(ours + llm)
# bias beyond chance; for fallacy oversight `value` is the share of correct reactions 1 - RR, and the bias is
# RR + CR - 1 = CR - value, the unchanged verdicts beyond chance
comparison["excess"] = np.where(comparison.bias == "fallacy", comparison.CR - comparison.value,
                                comparison.CR - comparison.value)
comparison.to_csv(TABLES / "comparison_with_calm.csv", index=False)
print("Robustness rate (fallacy: share of correct reactions): median and range over the judges of each kind;")
print("excess = median of CR - RR")
comparison.groupby(["bias", "variant", "kind"]).agg(median=("value", "median"), low=("value", "min"), high=("value", "max"),
                                                     excess=("excess", "median")).round(2).unstack("kind")

Robustness rate (fallacy: share of correct reactions): median and range over the judges of each kind;
excess = median of CR - RR


median              low             high           excess      
kind                   LLM (CALM) typed LLM (CALM) typed LLM (CALM) typed LLM (CALM) typed
bias        variant                                                                       
authority   Book             0.79  0.86       0.63  0.82       0.86  0.88       0.12  0.13
            Quote            0.75  0.81       0.66  0.75       0.86  0.87       0.14  0.19
            URL              0.83  0.87       0.70  0.83       0.88  0.91       0.06  0.11
bandwagon   60%              0.69  0.87       0.56  0.81       0.77  0.98       0.21  0.10
            70%              0.67  0.86       0.61  0.74       0.79  0.97       0.22  0.14
            80%              0.68  0.83       0.64  0.68       0.80  0.95       0.20  0.17
            90%              0.71  0.80       0.62  0.66       0.80  0.92       0.20  0.19
compassion  -                0.86  0.92       0.84  0.79       0.88  0.96       0.04  0.07
distraction h.c              0.75  0.96       0.72  0.88       0.90  0.98       0.15  0.03
            h.r              0.82  0.95       0.71  0.89       0.85  1.00       0.09  0.03
diversity   Black            0.82  0.96       0.66  0.87       0.90  0.99       0.09  0.04
            Female           0.82  0.96       0.65  0.89       0.92  0.99       0.10  0.02
            HIV Pos.         0.83  0.94       0.69  0.89       0.94  0.98       0.07  0.04
            Homosexual       0.82  0.95       0.70  0.88       0.94  0.99       0.09  0.03
            Muslim           0.82  0.95       0.71  0.89       0.91  0.98       0.08  0.04
            Refugees         0.81  0.95       0.67  0.88       0.90  0.98       0.09  0.04
fallacy     -                0.97  0.96       0.92  0.88       0.98  0.99       0.02  0.04
position    -                0.78  0.83       0.57  0.76       0.83  0.92       0.12  0.15
sentiment   Ang.(bet.)       0.32  0.38       0.24  0.21       0.64  0.66       0.67  0.62
            Ang.(wor.)       0.95  0.99       0.92  0.94       0.99  1.00       0.04  0.01
            Che.(bet.)       0.73  0.67       0.61  0.49       0.80  0.81       0.27  0.33
            Che.(wor.)       0.91  0.99       0.87  0.94       0.98  1.00       0.08  0.01
            Fea.(bet.)       0.27  0.27       0.24  0.07       0.64  0.36       0.72  0.73
            Fea.(wor.)       0.93  0.99       0.92  0.90       0.99  1.00       0.05  0.01
            Sad(bet.)        0.31  0.31       0.26  0.04       0.66  0.49       0.68  0.69
            Sad(wor.)        0.92  0.99       0.91  0.91       0.98  1.00       0.06  0.01
verbosity   -                0.91  0.99       0.88  0.90       0.98  1.00       0.08  0.01

## Formats compared (same model, same prompts)

In [ ]:
pairs = [("calm", "native"), ("native", "decomposed")]
cmp = pd.concat([metrics.compare_formats(tables[(m, a)], tables[(m, b)]).assign(model=short(m), formats=f"{a} vs {b}")
                 for m in run.MODELS for a, b in pairs if (m, a) in tables and (m, b) in tables], ignore_index=True)
cmp["significant"], cmp["p_adjusted"] = metrics.adjust(cmp.p_changed_verdicts)
cmp.to_csv(TABLES / "formats.csv", index=False)
print("number of biases (of 9) with a significant format difference in changed verdicts:")
display(cmp.groupby(["model", "formats"]).significant.sum().unstack())
cmp[cmp.significant][["model", "formats", "bias", "RR_a", "RR_b", "p_adjusted", "shift_diff", "shift_diff_lo", "shift_diff_hi"]].round(3)

number of biases (of 9) with a significant format difference in changed verdicts:


formats,calm vs native,native vs decomposed
model,,
clef,8,2
clef-flash,5,3
d1,2,0
jev-1.13,1,0
kev-4b,7,2
pplx-decider-v1.1-27b,2,0
solar-decide,2,3
tev1-4b,6,0


,model,formats,bias,RR_a,RR_b,p_adjusted,shift_diff,shift_diff_lo,shift_diff_hi
4,jev-1.13,calm vs native,distraction,0.933,0.954,0.040,-0.017,-0.020,-0.013
28,kev-4b,calm vs native,distraction,0.935,0.868,0.001,0.016,0.013,0.020
29,kev-4b,calm vs native,diversity,0.913,0.880,0.034,0.002,-0.001,0.004
30,kev-4b,calm vs native,"diversity, neutral note",0.946,0.894,0.009,0.006,0.003,0.009
31,kev-4b,calm vs native,fallacy,0.008,0.034,0.002,-0.075,-0.082,-0.069
33,kev-4b,calm vs native,sentiment,0.611,0.652,0.001,-0.055,-0.061,-0.048
34,kev-4b,calm vs native,verbosity,0.996,0.964,0.003,-0.011,-0.017,-0.005
35,kev-4b,calm vs native,"verbosity, better response",0.910,0.946,0.031,-0.063,-0.069,-0.057
43,kev-4b,native vs decomposed,fallacy,0.034,0.052,0.025,-0.011,-0.013,-0.010
47,kev-4b,native vs decomposed,"verbosity, better response",0.946,0.920,0.024,0.007,0.006,0.008


## Variants: majority share, citation type and side, tone and side, identity group vs neutral note

In [ ]:
var = pd.concat([metrics.summarize(df, by=["bias", "variant"], n_boot=2000).assign(model=short(m), format=f)
                 for (m, f), df in tables.items()], ignore_index=True)
var["significant"], var["p_adjusted"] = metrics.adjust(var.p_bias)
var.to_csv(TABLES / "variants.csv", index=False)
v = var[var.format == "calm"]
for bias in ["bandwagon", "authority", metrics.ADDED["authority"], "sentiment", "distraction"]:
    print(bias, "(CALM format): robustness rate and shift per variant")
    display(v[v.bias == bias].pivot(index="variant", columns="model", values="RR").round(2))
    display(v[v.bias == bias].pivot(index="variant", columns="model", values="shift").round(3))

bandwagon (CALM format): robustness rate and shift per variant


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
variant,,,,,,,,
60,0.81,0.93,0.96,0.82,0.87,0.84,0.87,0.98
70,0.74,0.89,0.93,0.78,0.84,0.74,0.89,0.97
80,0.68,0.87,0.92,0.77,0.79,0.72,0.88,0.95
90,0.68,0.84,0.91,0.76,0.73,0.66,0.84,0.92


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
variant,,,,,,,,
60,0.188,0.069,0.032,0.141,0.051,0.126,0.033,0.008
70,0.256,0.109,0.045,0.174,0.070,0.177,0.045,0.025
80,0.301,0.119,0.050,0.186,0.092,0.206,0.054,0.046
90,0.302,0.138,0.053,0.199,0.126,0.248,0.090,0.074


authority (CALM format): robustness rate and shift per variant


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
variant,,,,,,,,
book-on-R2,0.85,0.87,0.82,0.86,0.88,0.85,0.83,0.88
quote-on-R2,0.81,0.87,0.81,0.84,0.77,0.81,0.75,0.79
url-on-R2,0.84,0.90,0.83,0.91,0.89,0.85,0.86,0.89


...

model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
variant,,,,,,,,
angry-on-R1,0.44,0.66,0.23,0.21,0.29,0.31,0.55,0.51
angry-on-R2,1.00,0.94,1.00,1.00,0.99,1.00,0.97,0.99
cheerful-on-R1,0.72,0.81,0.50,0.49,0.55,0.66,0.72,0.69
cheerful-on-R2,1.00,0.94,1.00,1.00,0.98,1.00,0.94,0.99
fear-on-R1,0.24,0.31,0.22,0.16,0.07,0.30,0.36,0.34
fear-on-R2,1.00,0.90,1.00,1.00,0.99,1.00,0.96,0.99
sad-on-R1,0.28,0.35,0.26,0.20,0.04,0.39,0.49,0.42
sad-on-R2,1.00,0.91,1.00,1.00,0.99,1.00,0.97,0.99


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
variant,,,,,,,,
angry-on-R1,-0.521,-0.297,-0.625,-0.706,-0.393,-0.585,-0.418,-0.373
angry-on-R2,-0.016,-0.065,-0.076,-0.038,-0.135,-0.042,-0.035,-0.062
cheerful-on-R1,-0.281,-0.148,-0.396,-0.467,-0.276,-0.341,-0.252,-0.293
cheerful-on-R2,0.006,0.015,0.001,-0.018,-0.020,0.005,0.041,-0.025
fear-on-R1,-0.722,-0.558,-0.632,-0.732,-0.552,-0.593,-0.571,-0.450
fear-on-R2,-0.018,-0.100,-0.067,-0.037,-0.161,-0.038,-0.037,-0.058
sad-on-R1,-0.645,-0.553,-0.584,-0.676,-0.563,-0.530,-0.466,-0.431
sad-on-R2,-0.015,-0.099,-0.056,-0.034,-0.156,-0.029,-0.022,-0.048


distraction (CALM format): robustness rate and shift per variant


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
variant,,,,,,,,
about-R1,0.98,0.96,0.97,0.91,0.93,0.98,0.88,0.97
about-R2,1.00,0.96,0.97,0.94,0.94,0.98,0.89,0.94


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
variant,,,,,,,,
about-R1,-0.009,-0.017,-0.001,-0.069,-0.028,-0.008,-0.016,-0.021
about-R2,-0.007,-0.024,-0.007,-0.046,-0.028,-0.003,-0.002,-0.042


In [ ]:
# Diversity: shift of each identity note minus the shift of the neutral note on the same side (CALM format)
d = v[v.bias.isin(["diversity", metrics.ADDED["diversity"]])].copy()
d["group"] = d.variant.str.rsplit("-on-", n=1).str[0]
d["side"] = d.variant.str.rsplit("-on-", n=1).str[1]
neutral = d[d.group == "Software engineer"].set_index(["model", "side"])["shift"].rename("neutral_shift")
d = d.join(neutral, on=["model", "side"])
d["shift_minus_neutral"] = d["shift"] - d["neutral_shift"]
d[d.group != "Software engineer"].pivot_table(index=["group", "side"], columns="model", values="shift_minus_neutral").round(3)

model                   clef  clef-flash     d1  jev-1.13  kev-4b  pplx-decider-v1.1-27b  solar-decide  tev1-4b
group            side                                                                                          
Black            R1   -0.008       0.008  0.004     0.016  -0.013                 -0.001         0.030    0.008
                 R2   -0.010      -0.001 -0.000     0.013  -0.010                  0.002         0.013   -0.005
Female           R1    0.005       0.034  0.006     0.020  -0.004                  0.004         0.021    0.009
                 R2    0.005       0.025  0.001     0.013  -0.004                  0.003        -0.005   -0.000
HIV Positive     R1   -0.011       0.004  0.001    -0.016  -0.036                 -0.007         0.010   -0.008
                 R2   -0.015      -0.006 -0.005    -0.009  -0.032                 -0.001        -0.022   -0.019
Homosexual       R1   -0.008      -0.009  0.001     0.001  -0.047                 -0.002         0.017   -0.015
                 R2   -0.010      -0.015 -0.006     0.004  -0.039                  0.000         0.011   -0.026
Muslim believers R1   -0.011      -0.025 -0.001    -0.001  -0.030                 -0.007         0.007    0.002
                 R2   -0.012      -0.032 -0.006     0.001  -0.027                 -0.000        -0.012   -0.015
Refugees         R1   -0.013       0.010  0.003     0.000  -0.009                 -0.008         0.008   -0.003
                 R2   -0.019       0.002 -0.002    -0.001  -0.009                 -0.004        -0.021   -0.016

## Time and cost per model

In [ ]:
if SMOKE:
    print("time and cost are only meaningful for the full run")
else:
    log = pd.DataFrame([json.loads(l) for l in open(run.RESULTS / "run_log.jsonl")])
    log = log[(log.sent_now >= 100) & ~log.model.str.contains("mercury")]   # real sending passes, not cached replays
    passes = log.groupby(["model", "format"]).agg(wall_s=("wall_s", "sum"), passes=("wall_s", "size"),
                                                  concurrency=("concurrency", lambda x: "/".join(str(v) for v in x))).reset_index()
    passes["model"] = passes.model.map(short)
    per_req = pd.DataFrame([{"model": short(m), "format": f, "requests": len(recs),
                             "usd": sum(r["cost"] or 0 for r in recs),
                             "median_latency_s": np.median([r["latency_s"] for r in recs if r["ok"] and r["latency_s"]])}
                            for (m, f), recs in records.items() if recs and "mercury" not in m])
    cost = per_req.merge(passes, on=["model", "format"])
    cost["usd_per_1000"] = 1000 * cost.usd / cost.requests
    cost["wall_min"] = cost.wall_s / 60
    cost.to_csv(TABLES / "time_and_cost.csv", index=False)
    display(cost[["model", "format", "requests", "passes", "concurrency", "wall_min", "usd", "usd_per_1000", "median_latency_s"]].round(3))
    print("total: %.0f minutes, $%.2f" % (cost.wall_min.sum(), cost.usd.sum()))

,model,format,requests,passes,concurrency,wall_min,usd,usd_per_1000,median_latency_s
0,jev-1.13,calm,15195,2,32/16,2.248,0.677,0.045,0.252
1,jev-1.13,native,15195,2,32/16,3.547,0.683,0.045,0.343
2,jev-1.13,decomposed,15195,2,32/16,4.572,1.227,0.081,0.492
3,kev-4b,calm,15195,2,32/16,16.457,0.494,0.033,1.901
4,kev-4b,native,15195,2,32/16,20.330,0.479,0.032,1.862
5,kev-4b,decomposed,15195,2,32/16,23.313,0.892,0.059,2.763
6,tev1-4b,calm,15195,2,32/16,10.522,0.547,0.036,0.559
7,tev1-4b,native,15195,2,32/16,4.700,0.530,0.035,0.532
8,tev1-4b,decomposed,15195,2,32/16,10.755,3.117,0.205,1.249
9,solar-decide,calm,15195,2,32/16,46.597,0.836,0.055,2.866


total: 385 minutes, $37.96


## Notes

**Full run, 2026-10-04: 8 typed decision models x 3 formats x 14,695 requests, complete.** Mercury Decide is
excluded for now (free tier, 1,000 requests/day; decision pending). Tev1 gave one invalid reply ("D").
The statistics check finds the planted bias in 20/20 simulations with 2/160 false alarms.
All numbers below use the 50/50 rule (decided 2026-10-05, see the 50/50 bullet). Each bias pools CALM's versions
only (decided 2026-10-05): authority = citation on the worse response, diversity = the six identity notes on either
response. Our additions are tests of their own: `authority, better response` and `diversity, neutral note`
(264 tests in all, 242 significant).

* **Consistency.** CR 0.99 to 1.00 for seven models; Solar 0.93. Accuracy on the original alignment prompts
  0.65 to 0.78, on the fact items 0.95 to 1.00.
* **Shared by all 8 models (CALM format):** bandwagon pulls towards the worse response (shift +0.04 to +0.27,
  RR 0.61 to 0.96; strongest Perplexity and Clef); fake citations are *penalised*, on the worse response (shift -0.03 to -0.17) and even more on the better one
  (-0.04 to -0.31, RR down to 0.58);
  an emotional tone on the better response is penalised (shift -0.22 to -0.34); compassion fade towards the
  strong name in 7 of 8 (shift up to +0.11, Tev1). Position bias in all 8, but in both directions (six prefer
  the second response, D1 and Solar the first). Distraction and identity notes: small penalties. Verbosity:
  mostly robust (Clef Flash and Solar slightly prefer the padded answer; Tev1's tiny effect, excess 0.01, is
  significant only once 50/50 answers are left out). Fallacy: 88 to 99% correct reactions.
* **Against CALM's LLMs (2024), matched as closely as CALM reports** (Table 4 for position, verbosity,
  compassion, fallacy; Table 8 variant by variant for bandwagon, authority, sentiment, distraction, diversity).
  Per bias (each judge averaged over CALM's variants, then the median over judges), robustness rate typed vs LLM:
  verbosity 0.99 vs 0.91, distraction 0.96 vs 0.79, diversity 0.95 vs 0.82, bandwagon 0.84 vs 0.69, compassion 0.91
  vs 0.86, position 0.83 vs 0.78, authority 0.84 vs 0.79, sentiment 0.68 vs 0.67, fallacy (share correct) 0.96 vs 0.97.
  The typed models are more consistent (CR 0.99 to 1.00 against CALM's 0.86 to 0.93), so the fairer number is the
  excess CR - RR (medians): smaller for the typed models on verbosity 0.01 vs 0.08, distraction 0.04 vs 0.12,
  diversity 0.04 vs 0.09, bandwagon 0.15 vs 0.21; about equal on position 0.15 vs 0.12 and sentiment 0.32 vs 0.32;
  larger on authority 0.15 vs 0.11 (the typed models penalise the citation; CALM reports only how often LLM
  verdicts change, not in which direction) and compassion 0.08 vs 0.04.
* **Sentiment, variant by variant** (RR median typed vs LLM): an emotional tone on the better response is as harmful
  for the typed models as for the LLMs (sad 0.27 vs 0.31, angry 0.37 vs 0.32, fear 0.23 vs 0.27, cheerful 0.65 vs
  0.73); on the worse response the typed models barely react (0.99 for every tone vs 0.91 to 0.95).
  Bandwagon: the typed models' robustness falls with the claimed share (0.87 at 60% to 0.80 at 90%), the LLMs' does
  not (0.67 to 0.71). Distraction: the typed models react equally little on both sides (0.96 and 0.95), the LLMs more
  when the note is on the better response (0.75 vs 0.82).
* **Summary of the comparison:** typed decision models are not generally less biased than LLM judges. They are
  less swayed by padding, distraction and identity notes, about as much by position and emotional tone, and more
  by model names; citations sway them as much, but against the cited response.
* **50/50 answers (a finding, and a decision).** A typed model can answer exactly 50/50, and the API then still
  returns a `choice`. In the CALM format this happens for Tev1 in 410 of 14,695 answers (2.8%), D1 245, Solar 117,
  Jev 64, Perplexity 12, Kev 1, Clef and Clef Flash 0 (native about the same). Tev1, D1, Solar and Perplexity then
  *always* return "A", the option listed first; Jev and Kev return either. Jev rounds its probabilities to two
  decimals (101 distinct values), so its 50/50 answers are close calls. Tev1 also returns "A" for 30 answers like
  0.499994. Not documented: TypeSafe defines `choice` as "the option with the highest probability" and says nothing
  about ties; OpenRouter's API reference only lists the fields.
  **Decision (with the author, 2026-10-05):** an answer with |P(A) - 0.5| < 1e-5 has no verdict; it is left out of
  RR and CR but counts for the shifts, in all three formats. **Why:** counting the returned "A" would give an
  undecided judge a preference for the response shown first (in the position test, 50/50 in both orders would look
  like a changed verdict). **Effect:** RR changes by at most 0.03 (D1 compassion native, Tev1 identity notes and
  bandwagon), shifts in the CALM and native formats are unchanged, one test decision changes (Tev1 verbosity, CALM
  format, becomes significant), and one format difference appears (Jev distraction, CALM vs native). Tev1's CR
  becomes 1.00: its few repetition flips were all 50/50 answers. Written up in the paper (Section 3, Appendix E).
* **Confidence field** (not used, but a finding): TypeSafe's docs define confidence as 2 p_max - 1 (0 at 50/50). Jev, Kev,
  D1 and Perplexity follow it; Tev1 reports p_max (0.5 at 50/50); Solar reports 1 minus the entropy; the Clef models
  use a rule we could not identify. The same field means different things for different models.
* **Formats** matter for some models only: CALM vs native differs significantly in 1 (Jev, Perplexity) to 7 (Clef)
  of 11 tests; native vs decomposed in 0 to 3.
* **Time and cost** per model for 44,085 judgments: 10 min (Jev) to 170 min (Solar); Perplexity 87 min over two
  passes; $1.8 (Kev) to $12.3 (Clef) in total, i.e. $0.04 to $0.28 per 1,000 judgments; median latency per request
  0.2 to 3.4 s in the CALM and native formats, up to 12 s (Solar) with the six decomposed questions, which also
  cost 2 to 6 times as much. All 8 models together: 402 min and $39.18.